# B04A V2 — Koreksi audit crop citra film

**Mulai MODE='preview', runtime CPU, Run all.** Unduh dan kirim B04A_V2_preview_report.zip.
Jangan jalankan build atau B04B sebelum preview revisi diperiksa.

Audit V1 pada 12 contoh per dataset: CBIS-DDSM dan MIAS masing-masing 11/12 kotak tidak berubah.
Semua contoh kedua dataset memiliki faktor pembesaran 1,00 pada output persegi, walau sebagian luas kotak berkurang.
CMMD dan INbreast menunjukkan pemotongan latar belakang; bukan jaminan kualitas seluruh cohort.

V2 memperbarui penentuan kotak untuk CBIS-DDSM/MIAS. Pita luar 5% diabaikan **hanya dalam mask pencarian**,
kemudian tepi sumber dipertahankan bila komponen jaringan menyentuh batas pita buatan.
Ini **bukan pemotongan permanen 5% dari semua gambar**. Isi kotak memakai piksel sumber tanpa masking.
Ada margin dan fallback ke citra penuh bila pemeriksaan heuristik gagal. Efektivitas pada citra nyata belum diverifikasi.

CMMD dan INbreast memakai aturan crop V1. Semua dataset mendapat audit faktor pembesaran,
status kotak tidak berubah, versi library, dan thumbnail sumber terpisah (maksimum1024; bukan citra resolusi asli).
Orientasi MIAS dipertahankan. Citra melebar tidak otomatis berarti salah decode; crop tidak dipaksakan untuk memperbesar.
Tidak memakai ROI lesi, CLAHE, perubahan label, resplit, atau performa test untuk menentukan crop.

Kontrol full_224 tetap harus cocok piksel dengan B02C; full_512 tetap dibangun dari sumber asli.
Hash source baru membuat direktori B04A baru, sehingga cache lama tetap terpisah.
B04B dapat menggunakan registry V2 setelah READY. Bila lebih dari satu registry ada, isi B04A_REGISTRY dengan path V2.

INbreast tetap proxy BI-RADS dan grouping pseudonim belum terverifikasi. Test B03 sudah pernah dilihat.
Preview adalah audit teknis awal, bukan validasi klinis bahwa seluruh lesi terjaga.


## Cell 1 — Dependensi dan Drive


In [ ]:
import sys, subprocess
subprocess.check_call([sys.executable,'-m','pip','install','-q',
    'pydicom>=3,<4','pylibjpeg','pylibjpeg-libjpeg','pylibjpeg-openjpeg',
    'pillow','numpy','pandas','scipy','matplotlib'])
from google.colab import drive
drive.mount('/content/drive')


## Cell 2 — Implementasi preprocessing


In [ ]:
PREPARATION_SOURCE = 'import io, os, json, hashlib, zipfile, shutil, math\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\nfrom PIL import Image\nfrom scipy import ndimage\n\nVARIANTS = {\'full_224\':(False,224), \'breast_224\':(True,224),\n            \'full_512\':(False,512), \'breast_512\':(True,512)}\nIDENTITY = [\'sample_id\',\'patient_id\',\'label\',\'split\']\nCROP_PARAMETERS = dict(thumbnail=512, foreground_threshold=5, margin_fraction=.05,\n                       minimum_box_fraction=.15, maximum_outside_foreground=.03, film_detection_inset=.05)\n\ndef sha(data): return hashlib.sha256(data).hexdigest()\n\ndef file_sha(path):\n    h=hashlib.sha256()\n    with open(path,\'rb\') as f:\n        for block in iter(lambda:f.read(8*1024*1024),b\'\'): h.update(block)\n    return h.hexdigest()\n\ndef save_json(path, value):\n    path=Path(path); path.parent.mkdir(parents=True,exist_ok=True)\n    tmp=path.with_suffix(path.suffix+\'.partial\')\n    tmp.write_text(json.dumps(value,indent=2,allow_nan=False),encoding=\'utf-8\')\n    os.replace(tmp,path)\n\ndef identity_digest(frame):\n    return sha(frame[IDENTITY].sort_values(\'sample_id\').to_csv(index=False).encode())\n\ndef validate_manifest(frame):\n    needed=IDENTITY+[\'member\',\'source_file_sha256\',\'source_pixel_sha256\',\'processed_pixel_sha256\']\n    missing=set(needed)-set(frame)\n    if missing: raise ValueError(f\'Manifest missing {sorted(missing)}\')\n    for col in needed:\n        if frame[col].isna().any() or frame[col].astype(str).str.strip().eq(\'\').any():\n            raise ValueError(\'Empty manifest field: \'+col)\n    if frame.sample_id.duplicated().any(): raise ValueError(\'Duplicate sample IDs\')\n    if not frame.sample_id.str.fullmatch(\'[0-9a-fA-F]{64}\').all():\n        raise ValueError(\'Unexpected sample_id format; cache filenames must be SHA256\')\n    if not frame.label.isin([0,1]).all(): raise ValueError(\'Nonbinary labels\')\n    if set(frame.split)!={\'train\',\'validation\',\'test\'}: raise ValueError(\'Invalid split names\')\n    if frame.groupby(\'patient_id\').split.nunique().max()!=1: raise ValueError(\'Patient group overlap\')\n    for _,g in frame.groupby(\'split\'):\n        if set(g.label)!={0,1}: raise ValueError(\'A split is missing a class\')\n    for col in [\'source_pixel_sha256\',\'processed_pixel_sha256\']:\n        g=frame.groupby(col).agg(splits=(\'split\',\'nunique\'),labels=(\'label\',\'nunique\'))\n        if ((g.splits>1)|(g.labels>1)).any(): raise ValueError(\'Existing duplicate conflict: \'+col)\n\ndef read_manifest(entry):\n    if entry.get(\'status\')!=\'READY\': raise RuntimeError(\'B02C entry is not READY\')\n    folder=Path(entry[\'path\']); meta=json.loads((folder/\'READY.json\').read_text())\n    frames=[]\n    for split in [\'train\',\'validation\',\'test\']:\n        p=folder/(split+\'.csv\')\n        if file_sha(p)!=meta[\'file_sha256\'][p.name]: raise RuntimeError(\'B02C CSV hash changed: \'+str(p))\n        f=pd.read_csv(p,dtype={\'sample_id\':str,\'patient_id\':str},keep_default_na=False)\n        if not f.split.eq(split).all(): raise ValueError(\'Split column disagrees with CSV name\')\n        # Validate numeric labels before casting, to avoid truncating invalid fractions.\n        f[\'label\']=pd.to_numeric(f.label,errors=\'raise\')\n        if not f.label.isin([0,1]).all(): raise ValueError(\'Invalid label\')\n        f[\'label\']=f.label.astype(int); frames.append(f)\n    frame=pd.concat(frames,ignore_index=True)\n    validate_manifest(frame)\n    if identity_digest(frame)!=meta[\'cohort_sha256\']: raise RuntimeError(\'Cohort identity hash changed\')\n    return frame,meta\n\ndef decode_original(raw, member):\n    # Exactly the B02 decoding policy: modality LUT, per-image min/max and MONOCHROME1 inversion.\n    if member.lower().endswith(\'.dcm\'):\n        import pydicom\n        from pydicom.pixels import apply_modality_lut\n        ds=pydicom.dcmread(io.BytesIO(raw))\n        photo=str(ds.get(\'PhotometricInterpretation\',\'\'))\n        if photo not in [\'MONOCHROME1\',\'MONOCHROME2\']: raise ValueError(\'Unsupported DICOM photometric interpretation\')\n        pixels=np.asarray(apply_modality_lut(ds.pixel_array,ds)); invert=photo==\'MONOCHROME1\'\n    else:\n        with Image.open(io.BytesIO(raw)) as im: pixels=np.asarray(im.convert(\'L\'))\n        invert=False\n    if pixels.ndim!=2: raise ValueError(\'Expected one grayscale frame\')\n    digest=sha(str((pixels.shape,str(pixels.dtype))).encode()+pixels.tobytes())\n    a=pixels.astype(np.float32)\n    if not np.isfinite(a).all() or a.max()<=a.min(): raise ValueError(\'Constant/nonfinite image\')\n    lo,hi=float(a.min()),float(a.max()); a=(a-lo)/(hi-lo)\n    if invert: a=1-a\n    return np.rint(np.clip(a,0,1)*255).astype(np.uint8),digest\n\ndef letterbox(a,size):\n    h,w=a.shape; scale=min(size/w,size/h)\n    nw,nh=max(1,round(w*scale)),max(1,round(h*scale))\n    small=Image.fromarray(a).resize((nw,nh),Image.Resampling.BILINEAR)\n    canvas=Image.new(\'L\',(size,size),0)\n    canvas.paste(small,((size-nw)//2,(size-nh)//2))\n    return np.asarray(canvas).copy()\n\ndef breast_box_legacy(a, params=CROP_PARAMETERS):\n    # Conservative background bounding box, NOT a lesion annotation or tissue segmentation.\n    # No pixels inside the bounding rectangle are masked out.\n    h,w=a.shape; scale=min(1.,params[\'thumbnail\']/max(h,w))\n    th,tw=max(1,round(h*scale)),max(1,round(w*scale))\n    small=np.asarray(Image.fromarray(a).resize((tw,th),Image.Resampling.BILINEAR))\n    foreground=small>params[\'foreground_threshold\']\n    components,n=ndimage.label(foreground,structure=np.ones((3,3)))\n    reason=\'\'; proposed=(0,0,w,h); outside=0.\n    if n==0: reason=\'no_component\'\n    else:\n        sizes=np.bincount(components.ravel()); sizes[0]=0\n        yy,xx=np.where(components==sizes.argmax())\n        margin_x=max(2,math.ceil(tw*params[\'margin_fraction\']))\n        margin_y=max(2,math.ceil(th*params[\'margin_fraction\']))\n        x0=max(0,int(xx.min())-margin_x); x1=min(tw,int(xx.max())+1+margin_x)\n        y0=max(0,int(yy.min())-margin_y); y1=min(th,int(yy.max())+1+margin_y)\n        outside=float((foreground.sum()-foreground[y0:y1,x0:x1].sum())/max(1,foreground.sum()))\n        proposed=(math.floor(x0*w/tw),math.floor(y0*h/th),math.ceil(x1*w/tw),math.ceil(y1*h/th))\n        if (x1-x0)*(y1-y0)/(tw*th)<params[\'minimum_box_fraction\']: reason=\'box_too_small\'\n        elif outside>params[\'maximum_outside_foreground\']: reason=\'foreground_outside_box\'\n    box=(0,0,w,h) if reason else proposed\n    x0,y0,x1,y1=box\n    return box,dict(crop_x0=x0,crop_y0=y0,crop_x1=x1,crop_y1=y1,\n        crop_fraction=(x1-x0)*(y1-y0)/(h*w),outside_foreground_fraction=outside,\n        crop_fallback=bool(reason),crop_note=reason or \'automatic_box_requires_visual_review\')\n\ndef breast_box(a, params=CROP_PARAMETERS, dataset=\'\'):\n    """V2: ignore a narrow film frame only in the detection mask, never in source pixels."""\n    old_box,old=breast_box_legacy(a,params)\n    h,w=a.shape\n    if dataset not in (\'cbis_ddsm\',\'mias\'):\n        box,audit=old_box,dict(old)\n        audit[\'crop_method\']=\'legacy_low_threshold_component\'\n    else:\n        scale=min(1.,params[\'thumbnail\']/max(h,w))\n        th,tw=max(1,round(h*scale)),max(1,round(w*scale))\n        small=np.asarray(Image.fromarray(a).resize((tw,th),Image.Resampling.BILINEAR))\n        by=max(1,int(math.ceil(th*params[\'film_detection_inset\'])))\n        bx=max(1,int(math.ceil(tw*params[\'film_detection_inset\'])))\n        fg=small>params[\'foreground_threshold\']\n        interior=np.zeros_like(fg)\n        interior[by:th-by,bx:tw-bx]=True\n        candidates=ndimage.binary_opening(fg & interior,structure=np.ones((3,3)),iterations=1)\n        components,n=ndimage.label(candidates,structure=np.ones((3,3)))\n        reason=\'\';outside=0.;candidate=(0,0,w,h)\n        if n==0:reason=\'no_interior_component\'\n        else:\n            sizes=np.bincount(components.ravel());sizes[0]=0\n            yy,xx=np.where(components==sizes.argmax())\n            x0,x1=int(xx.min()),int(xx.max())+1\n            y0,y1=int(yy.min()),int(yy.max())+1\n            # If tissue meets the artificial inset boundary, retain the original image edge.\n            # This avoids automatically removing 5% of actual breast tissue.\n            if x0<=bx+2:x0=0\n            if x1>=tw-bx-2:x1=tw\n            if y0<=by+2:y0=0\n            if y1>=th-by-2:y1=th\n            mx=max(2,math.ceil(tw*params[\'margin_fraction\']))\n            my=max(2,math.ceil(th*params[\'margin_fraction\']))\n            x0,x1=max(0,x0-mx),min(tw,x1+mx)\n            y0,y1=max(0,y0-my),min(th,y1+my)\n            # QC uses the interior mask; the ignored film frame is not considered missing tissue.\n            valid_fg=fg & interior\n            outside=float((valid_fg.sum()-valid_fg[y0:y1,x0:x1].sum())/max(1,valid_fg.sum()))\n            candidate=(math.floor(x0*w/tw),math.floor(y0*h/th),\n                       min(w,math.ceil(x1*w/tw)),min(h,math.ceil(y1*h/th)))\n            if (x1-x0)*(y1-y0)/(tw*th)<params[\'minimum_box_fraction\']:reason=\'box_too_small\'\n            elif outside>params[\'maximum_outside_foreground\']:reason=\'interior_foreground_outside_box\'\n        box=(0,0,w,h) if reason else candidate\n        x0,y0,x1,y1=box\n        audit=dict(crop_x0=x0,crop_y0=y0,crop_x1=x1,crop_y1=y1,\n            crop_fraction=(x1-x0)*(y1-y0)/(h*w),outside_foreground_fraction=outside,\n            crop_fallback=bool(reason),crop_note=reason or \'film_box_requires_visual_review\',\n            crop_method=\'film_interior_component_v2\')\n    x0,y0,x1,y1=box\n    gain=max(h,w)/max(y1-y0,x1-x0)\n    audit.update(old_crop_fraction=old[\'crop_fraction\'],scale_gain_same_output=float(gain),\n        old_scale_gain_same_output=float(max(h,w)/max(old_box[3]-old_box[1],old_box[2]-old_box[0])),\n        crop_unchanged=bool(box==(0,0,w,h)),\n        no_material_scale_gain=bool(gain<1.02),\n        qc_scope=\'heuristic foreground only; tissue/lesion retention requires visual review\')\n    return box,audit\n\n\n\ndef checked_image(src,row):\n    raw=src.read(row[\'member\'])  # ZIP CRC is also checked by zipfile.\n    if sha(raw)!=row[\'source_file_sha256\']: raise RuntimeError(\'Source file hash mismatch: \'+row[\'sample_id\'])\n    a,digest=decode_original(raw,row[\'member\'])\n    if digest!=row[\'source_pixel_sha256\']: raise RuntimeError(\'Source pixel hash mismatch\')\n    full224=letterbox(a,224)\n    if sha(full224.tobytes())!=row[\'processed_pixel_sha256\']:\n        raise RuntimeError(\'Control 224 pixels do not match B02C. Stop before running ablation.\')\n    return a,full224\n\ndef validate_archive(src,frame):\n    names=src.namelist(); required=set(frame.member)\n    counts={name:0 for name in required}\n    for name in names:\n        if name in counts: counts[name]+=1\n    bad=[name for name,count in counts.items() if count!=1]\n    if bad: raise RuntimeError(\'Missing/ambiguous ZIP member: \'+str(bad[:3]))\n\ndef stage_selected_sources(archive,frame,cache_root):\n    """Copy only cohort members, not the entire source dataset archive, to Colab disk."""\n    cache_root.mkdir(parents=True,exist_ok=True)\n    signature=sha(frame[[\'member\',\'source_file_sha256\']].sort_values(\'member\').to_csv(index=False).encode())[:20]\n    target=cache_root/(signature+\'.zip\'); marker=cache_root/(signature+\'.json\')\n    if target.exists() and marker.exists():\n        data=json.loads(marker.read_text())\n        if file_sha(target)==data[\'sha256\']: return target\n        raise RuntimeError(\'Corrupted local source cache; delete that cache before retrying\')\n    with zipfile.ZipFile(archive) as src:\n        validate_archive(src,frame)\n        needed=sum(src.getinfo(m).file_size for m in set(frame.member))\n        if shutil.disk_usage(cache_root).free<needed+3*1024**3:\n            raise RuntimeError(\'Insufficient local disk for selected sources. Set STAGE_SOURCES=False to read directly from Drive.\')\n        partial=target.with_suffix(\'.partial\')\n        with zipfile.ZipFile(partial,\'w\',zipfile.ZIP_STORED) as dst:\n            for i,row in enumerate(frame.drop_duplicates(\'member\').to_dict(\'records\')):\n                raw=src.read(row[\'member\'])\n                if sha(raw)!=row[\'source_file_sha256\']:raise RuntimeError(\'Source changed\')\n                dst.writestr(row[\'member\'],raw)\n                if (i+1)%200==0:print(\'Staging selected original files:\',i+1,flush=True)\n    os.replace(partial,target);save_json(marker,{\'sha256\':file_sha(target)})\n    return target\n\ndef preview_rows(frame,per_class=3):\n    # Deterministic preview from TRAIN/VALIDATION only; never select by performance.\n    return pd.concat([g.sort_values(\'sample_id\').head(per_class)\n        for _,g in frame[frame.split.ne(\'test\')].groupby([\'split\',\'label\'])],ignore_index=True)\n\ndef annotation_inventory(src,dataset):\n    records=[]\n    for item in src.infolist():\n        name=item.filename; low=name.lower()\n        if item.is_dir(): continue\n        if low.endswith((\'.csv\',\'.xml\',\'.json\',\'.txt\',\'.xlsx\')) or \'roi\' in low or \'mask\' in low:\n            records.append(dict(dataset=dataset,member=name,bytes=item.file_size,\n                status=\'candidate_file_only; annotation mapping not yet verified\'))\n    return pd.DataFrame(records,columns=[\'dataset\',\'member\',\'bytes\',\'status\'])\n\ndef export_preview(frame,meta,archive,out):\n    import matplotlib.pyplot as plt\n    from matplotlib.patches import Rectangle\n    out.mkdir(parents=True,exist_ok=True)\n    selected=preview_rows(frame); records=[]\n    with zipfile.ZipFile(archive) as src:\n        validate_archive(src,frame)\n        annotation_inventory(src,meta[\'dataset\']).to_csv(out/\'annotation_inventory.csv\',index=False)\n        for page,start in enumerate(range(0,len(selected),4),1):\n            group=selected.iloc[start:start+4]\n            fig,axes=plt.subplots(len(group),5,figsize=(15,3.7*len(group)),squeeze=False)\n            for axes_row,row in zip(axes,group.to_dict(\'records\')):\n                a,control=checked_image(src,row); box,audit=breast_box(a,dataset=meta[\'dataset\'])\n                x0,y0,x1,y1=box; crop=a[y0:y1,x0:x1]\n                thumb=Image.fromarray(a);thumb.thumbnail((1024,1024),Image.Resampling.BILINEAR)\n                thumb_dir=out/\'source_thumbnails\';thumb_dir.mkdir(exist_ok=True)\n                thumb.save(thumb_dir/(row[\'sample_id\']+\'.png\'))\n                views=[a,control,letterbox(crop,224),letterbox(a,512),letterbox(crop,512)]\n                titles=[\'Original + proposed box\',\'Full 224\',\'Breast 224\',\'Full 512\',\n                        \'Breast 512 | scale x\'+format(audit[\'scale_gain_same_output\'],\'.2f\')]\n                for ax,im,title in zip(axes_row,views,titles):\n                    ax.imshow(im,cmap=\'gray\',vmin=0,vmax=255); ax.set_title(title);ax.axis(\'off\')\n                axes_row[0].add_patch(Rectangle((x0,y0),x1-x0,y1-y0,fill=False,edgecolor=\'lime\',linewidth=1))\n                axes_row[0].set_title(f"{row[\'split\']} | label {row[\'label\']} | {row[\'sample_id\'][:8]}")\n                records.append(dict(sample_id=row[\'sample_id\'],split=row[\'split\'],label=int(row[\'label\']),\n                    original_height=a.shape[0],original_width=a.shape[1],**audit))\n            fig.suptitle(meta[\'dataset\']+\' | preprocessing audit (train/validation)\',fontsize=14)\n            fig.tight_layout();fig.savefig(out/f\'preview_{page:02d}.png\',dpi=130);plt.show();plt.close(fig)\n    pd.DataFrame(records).to_csv(out/\'preview_audit.csv\',index=False)\n    import platform,importlib.metadata\n    versions={}\n    for package in [\'numpy\',\'pandas\',\'Pillow\',\'scipy\',\'pydicom\']:\n        try:versions[package]=importlib.metadata.version(package)\n        except importlib.metadata.PackageNotFoundError:versions[package]=\'not_installed\'\n    save_json(out/\'environment.json\',dict(python=platform.python_version(),versions=versions))\n    return records\n\ndef build_variant(frame,meta,archive,root,variant,source_hash):\n    crop_enabled,size=VARIANTS[variant]\n    signature=sha(json.dumps(dict(cohort=identity_digest(frame),parent_files=meta[\'file_sha256\'],variant=variant,\n        source_hash=source_hash,crop_parameters=CROP_PARAMETERS),sort_keys=True).encode())[:16]\n    dest=root/meta[\'dataset\']/variant/signature;dest.mkdir(parents=True,exist_ok=True)\n    marker=dest/\'READY.json\'\n    if marker.exists():\n        saved=json.loads(marker.read_text())\n        for name,digest in saved[\'file_sha256\'].items():\n            if file_sha(dest/name)!=digest: raise RuntimeError(\'Existing B04 cache hash mismatch\')\n        return dict(status=\'READY\',path=str(dest),**saved)\n    partial=dest/\'images.zip.partial\'; records=[]\n    with zipfile.ZipFile(archive) as src,zipfile.ZipFile(partial,\'w\',zipfile.ZIP_STORED) as dst:\n        validate_archive(src,frame)\n        for i,row in enumerate(frame.to_dict(\'records\')):\n            a,control=checked_image(src,row);box,audit=breast_box(a,dataset=meta[\'dataset\'])\n            x0,y0,x1,y1=box\n            result=letterbox(a[y0:y1,x0:x1] if crop_enabled else a,size)\n            if variant==\'full_224\' and not np.array_equal(result,control): raise RuntimeError(\'Invalid control\')\n            b=io.BytesIO();Image.fromarray(result).save(b,format=\'PNG\')\n            name=row[\'sample_id\']+\'.png\';dst.writestr(name,b.getvalue())\n            records.append(dict(row,cache_member=name,b02_processed_pixel_sha256=row[\'processed_pixel_sha256\'],\n                processed_pixel_sha256=sha(result.tobytes()),variant=variant,crop_applied=crop_enabled,\n                original_height=a.shape[0],original_width=a.shape[1],**audit))\n            if (i+1)%100==0 or i+1==len(frame): print(meta[\'dataset\'],variant,i+1,\'/\',len(frame),flush=True)\n    result=pd.DataFrame(records)\n    if identity_digest(result)!=identity_digest(frame): raise RuntimeError(\'Cohort changed\')\n    result.to_csv(dest/\'all.csv\',index=False)\n    # Crop/resize can create new identical inputs. Do not quietly remove rows or alter splits.\n    try: validate_manifest(result)\n    except Exception:\n        result.to_csv(dest/\'CONFLICT_REVIEW.csv\',index=False); raise\n    for split,g in result.groupby(\'split\'):g.to_csv(dest/(split+\'.csv\'),index=False)\n    result[[\'sample_id\',\'split\',\'crop_applied\',\'crop_fraction\',\'crop_fallback\',\'crop_note\',\'crop_method\',\'scale_gain_same_output\',\'old_scale_gain_same_output\',\'crop_unchanged\',\'no_material_scale_gain\',\n            \'outside_foreground_fraction\',\'original_height\',\'original_width\']].to_csv(dest/\'crop_audit.csv\',index=False)\n    os.replace(partial,dest/\'images.zip\')\n    saved=dict(dataset=meta[\'dataset\'],pipeline_version=\'b04_\'+variant+\'_film_box_v2\',\n        variant=variant,image_size=size,cohort_sha256=identity_digest(frame),\n        preprocessing_source_sha256=source_hash,crop_parameters=CROP_PARAMETERS,\n        source_archive=str(archive),parent_b02_metadata=meta,\n        images=len(frame),patient_groups=int(frame.patient_id.nunique()),split_seed=meta.get(\'split_seed\',2026),\n        grouping_caveat=meta.get(\'grouping_caveat\',\'\'),\n        split_policy=\'exactly retained B02C sample IDs, labels and split assignments; test previously inspected in B03\',\n        duplicates_checked=\'exact source and processed pixels; near duplicates not established\',\n        file_sha256={name:file_sha(dest/name) for name in [\'all.csv\',\'train.csv\',\'validation.csv\',\'test.csv\',\'images.zip\',\'crop_audit.csv\']})\n    save_json(marker,saved)\n    return dict(status=\'READY\',path=str(dest),**saved)\n\ndef audit_summary(frame,key):\n    return [dict(dataset=key,split=s,images=len(g),patient_groups=int(g.patient_id.nunique()),\n        negative=int(g.label.eq(0).sum()),positive=int(g.label.eq(1).sum()),\n        positive_groups=int(g.loc[g.label.eq(1),\'patient_id\'].nunique()),\n        label_source=\' | \'.join(sorted(set(g.get(\'label_source\',pd.Series([\'unspecified\'])).astype(str)))),\n        group_source=\' | \'.join(sorted(set(g.get(\'patient_group_source\',pd.Series([\'unspecified\'])).astype(str)))))\n        for s,g in frame.groupby(\'split\')]\n'
exec(compile(PREPARATION_SOURCE,'<B04A_V2>','exec'))
PREPARATION_SHA256=sha(PREPARATION_SOURCE.encode())
print('Preprocessing SHA256:',PREPARATION_SHA256)


## Cell 3 — Konfigurasi
Preview pada empat dataset. Build satu dataset per sesi agar ruang disk terkontrol.
STAGE_SOURCES=True menyalin hanya berkas cohort ke disk Colab, bukan seluruh ZIP dataset.


In [ ]:
RESEARCH=Path('/content/drive/MyDrive/Research')
MODE='preview'  # 'preview' atau 'build'
DATASETS_TO_PREVIEW=['cbis_ddsm','cmmd','inbreast','mias']
DATASETS_TO_BUILD=['cbis_ddsm']
VARIANTS_TO_BUILD=list(VARIANTS)
STAGE_SOURCES=True
SOURCE_OVERRIDES={}  # Optional: {'cbis_ddsm': '/content/.../original.zip'}
PREP_B02=RESEARCH/'outputs/benchmark_4datasets/prepared_v1'
OUTPUT_ROOT=RESEARCH/'outputs/benchmark_4datasets/b04_preprocessing_v1'/PREPARATION_SHA256[:12]
LOCAL_SOURCE_CACHE=Path('/content/b04_selected_original_files')
OUTPUT_ROOT.mkdir(parents=True,exist_ok=True)
if MODE not in ['preview','build']:raise ValueError('Invalid MODE')
if not VARIANTS_TO_BUILD or not set(VARIANTS_TO_BUILD)<=set(VARIANTS):raise ValueError('Invalid variants')
registry_path=PREP_B02/'registry.json'
B02_REGISTRY=json.loads(registry_path.read_text())
(OUTPUT_ROOT/'preparation_source.py').write_text(PREPARATION_SOURCE,encoding='utf-8')
save_json(OUTPUT_ROOT/'session_config.json',dict(mode=MODE,preview_datasets=DATASETS_TO_PREVIEW,
    build_datasets=DATASETS_TO_BUILD,variants=VARIANTS_TO_BUILD,source_sha256=PREPARATION_SHA256,
    b02_registry_sha256=file_sha(registry_path),crop_parameters=CROP_PARAMETERS))
print('B04 output:',OUTPUT_ROOT)


## Cell 4 — Audit cohort dan gambar
Contoh deterministik dari setiap kelas pada train/validation, bukan berdasarkan kesalahan model.
Jumlah test ditampilkan untuk audit manifest, bukan pemilihan preprocessing.
Inventaris anotasi hanya daftar kandidat berkas; belum membuktikan ROI dapat dipasangkan.


In [ ]:
ACTIVE_DATASETS=DATASETS_TO_PREVIEW if MODE=='preview' else DATASETS_TO_BUILD
SUMMARIES=[];AUDIT_ERRORS=[];LOADED={}
for key in ACTIVE_DATASETS:
    try:
        frame,meta=read_manifest(B02_REGISTRY[key])
        archive=Path(SOURCE_OVERRIDES.get(key,meta['source_archive']))
        if not archive.is_file():raise FileNotFoundError(f'ZIP asli tidak ditemukan: {archive}')
        print('AUDIT',key,flush=True)
        cohort_out=OUTPUT_ROOT/'audit'/key/meta['cohort_sha256'][:16]
        export_preview(frame,meta,archive,cohort_out)
        SUMMARIES.extend(audit_summary(frame,key))
        LOADED[key]=(frame,meta,archive)
        save_json(cohort_out/'provenance.json',dict(parent_b02=meta,source_archive=str(archive),
            preparation_source_sha256=PREPARATION_SHA256,preview_splits=['train','validation']))
    except Exception as e:
        AUDIT_ERRORS.append(dict(dataset=key,error=type(e).__name__+': '+str(e)))
        print('AUDIT FAILED:',key,str(e),flush=True)
summary=pd.DataFrame(SUMMARIES)
summary.to_csv(OUTPUT_ROOT/'cohort_summary.csv',index=False)
save_json(OUTPUT_ROOT/'audit_errors.json',AUDIT_ERRORS)
display(summary)
if AUDIT_ERRORS:display(pd.DataFrame(AUDIT_ERRORS))
print('INbreast: BI-RADS proxy; grouping pasien dari nama berkas belum terverifikasi.')


## Cell 5 — Build ketika MODE='build'
Periksa preview dahulu. Byte berkas asli dan piksel diverifikasi terhadap manifest B02C.
Jika terputus, ulangi notebook: varian READY digunakan ulang, varian belum selesai dibangun kembali.
Konflik baru akibat crop/resize akan menghentikan varian; jangan mengubah label/split untuk meloloskannya.


In [ ]:
B04_REGISTRY_PATH=OUTPUT_ROOT/'registry.json'
B04_REGISTRY=json.loads(B04_REGISTRY_PATH.read_text()) if B04_REGISTRY_PATH.exists() else {}
BUILD_ERRORS=[]
if MODE=='build':
    for key in DATASETS_TO_BUILD:
        if key not in LOADED:
            BUILD_ERRORS.append(dict(dataset=key,error='Preview/source audit failed'));continue
        frame,meta,archive=LOADED[key]
        try:
            working=stage_selected_sources(archive,frame,LOCAL_SOURCE_CACHE) if STAGE_SOURCES else archive
            for variant in VARIANTS_TO_BUILD:
                entry=build_variant(frame,meta,working,OUTPUT_ROOT/'prepared',variant,PREPARATION_SHA256)
                B04_REGISTRY.setdefault(key,{})[variant]=entry
                save_json(B04_REGISTRY_PATH,B04_REGISTRY)
            print('READY:',key,flush=True)
        except Exception as e:
            BUILD_ERRORS.append(dict(dataset=key,error=type(e).__name__+': '+str(e)))
            print('BUILD FAILED:',key,str(e),flush=True)
    save_json(OUTPUT_ROOT/'build_errors.json',BUILD_ERRORS)
else:
    print('Preview selesai; cache belum dibangun. Periksa PNG, lalu MODE=build dan jalankan ulang.')
print('Registry untuk B04B:',B04_REGISTRY_PATH)


In [ ]:
EFFECT_ROWS=[]
for key,(frame,meta,archive) in LOADED.items():
    p=OUTPUT_ROOT/'audit'/key/meta['cohort_sha256'][:16]/'preview_audit.csv'
    d=pd.read_csv(p)
    EFFECT_ROWS.append(dict(dataset=key,preview_images=len(d),
        unchanged=int(d.crop_unchanged.sum()),fallback=int(d.crop_fallback.sum()),
        median_scale_gain=float(d.scale_gain_same_output.median()),
        maximum_scale_gain=float(d.scale_gain_same_output.max()),
        no_material_scale_gain=int(d.no_material_scale_gain.sum())))
effects=pd.DataFrame(EFFECT_ROWS)
effects.to_csv(OUTPUT_ROOT/'preview_effect_summary.csv',index=False)
display(effects)
print('Scale x1.00 = detail tidak diperbesar pada output persegi; hanya framing/posisi yang dapat berubah.')


## Cell 6 — Unduh laporan kecil


In [ ]:
report=OUTPUT_ROOT/f'B04A_V2_{MODE}_report.zip'
with zipfile.ZipFile(report,'w',zipfile.ZIP_DEFLATED) as z:
    for p in OUTPUT_ROOT.rglob('*'):
        if p.is_file() and p.suffix in ['.json','.csv','.py','.png']:
            z.write(p,str(p.relative_to(OUTPUT_ROOT)))
print(report)
from google.colab import files
files.download(str(report))
if AUDIT_ERRORS or BUILD_ERRORS:print('Ada error dalam laporan. Dataset gagal belum boleh masuk training.')
